# 01 · LoRA — Low-Rank Adaptation for Efficient Fine-Tuning

> **Module:** 07 – Fine-tuning  
> **Objective:** Understand and implement LoRA from first principles, then apply it to fine-tune a language model with a fraction of the parameters.

---

## 🎯 Learning Objectives

1. Explain why full fine-tuning of large models is impractical
2. Derive the LoRA low-rank decomposition mathematically
3. Implement `LoRALinear` from scratch in PyTorch
4. Apply LoRA to a transformer model using HuggingFace PEFT
5. Understand rank, alpha, and target module choices
6. Merge LoRA weights back into the base model for zero-overhead inference

---

## 1. The Problem: Full Fine-Tuning is Expensive

Fine-tuning LLaMA-3-70B from scratch:

```
Model parameters: 70B × 4 bytes (fp32) = 280 GB
Optimizer states: 280 GB × 2 (Adam) = 560 GB
Gradients:        280 GB
Total:            ~1.1 TB GPU memory

→ Requires ~14 × 80GB A100s just for training.
→ At $3/hr per GPU: $42/hr, weeks of training = $100k+
```

**Key Insight from LoRA paper (Hu et al., 2021):**
> "The weight matrices in pre-trained language models have low intrinsic rank when adapting to downstream tasks."

We don't need to update all 70B parameters. The **task-specific adaptation** can be expressed in a very low-dimensional subspace.

---

## 2. LoRA Mathematics

For a pre-trained weight matrix `W ∈ ℝ^(d×k)`, LoRA constrains the update:

```
Full fine-tuning:  W' = W + ΔW         (d×k parameters to update)

LoRA:              W' = W + BA          where B ∈ ℝ^(d×r), A ∈ ℝ^(r×k)
                                         r << min(d, k)
```

**Parameter savings:**
```
Full ΔW:  d × k parameters  (e.g., 4096 × 4096 = 16.7M)
LoRA:     r × (d + k) params (e.g., 16 × (4096 + 4096) = 131K)

Reduction: 16.7M / 131K ≈ 127x fewer parameters! 🔥
```

**Scaling factor α:**
```
output = W·x + (α/r) · B·A·x
```
Setting `α = 2r` is a common default. α controls the learning rate scale of the adapter.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install transformers peft datasets trl bitsandbytes -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
from typing import Optional

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 3. LoRA: From Scratch Implementation

In [ ]:
class LoRALinear(nn.Module):
    """
    Drop-in replacement for nn.Linear with LoRA adaptation.
    
    Usage:
        # Replace any linear layer with LoRA:
        original = nn.Linear(4096, 4096)
        lora_layer = LoRALinear(4096, 4096, rank=16, alpha=32)
        
        # Only A and B are trained; W_frozen stays fixed.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        rank: int = 16,
        alpha: float = 32.0,
        dropout: float = 0.0,
        bias: bool = True,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank  # scale factor applied to BA

        # Frozen pre-trained weights (not updated during training)
        self.weight = nn.Parameter(
            torch.empty(out_features, in_features),
            requires_grad=False  # ← FROZEN
        )
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=False)
        else:
            self.bias = None

        # LoRA parameters (trainable)
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))   # A: (r × k)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))  # B: (d × r)
        # Note: B initialized to zero → at step 0, LoRA contributes nothing
        # A initialized with Kaiming uniform (standard practice)

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # Weight initialization
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        # lora_B stays zero-initialized

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Base model output (frozen)
        base_out = F.linear(x, self.weight, self.bias)

        # LoRA delta: B·A·x scaled by α/r
        lora_out = F.linear(self.dropout(x), self.lora_A)   # (batch, seq, r)
        lora_out = F.linear(lora_out, self.lora_B)           # (batch, seq, d)
        lora_out = lora_out * self.scaling

        return base_out + lora_out

    def merge_weights(self) -> nn.Linear:
        """
        Merge LoRA weights into the base weight for zero-overhead inference.
        After merging, this layer behaves like a plain nn.Linear.
        """
        merged = nn.Linear(self.in_features, self.out_features,
                           bias=self.bias is not None)
        with torch.no_grad():
            delta_W = (self.lora_B @ self.lora_A) * self.scaling  # (d × k)
            merged.weight.copy_(self.weight + delta_W)
            if self.bias is not None:
                merged.bias.copy_(self.bias)
        return merged

    def trainable_parameters(self) -> int:
        return self.lora_A.numel() + self.lora_B.numel()

    def total_parameters(self) -> int:
        return self.weight.numel() + self.lora_A.numel() + self.lora_B.numel()


# Verify implementation
layer = LoRALinear(4096, 4096, rank=16, alpha=32)
x = torch.randn(2, 64, 4096)  # (batch, seq, d)
out = layer(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"\nParameter counts:")
print(f"  Base weight:   {layer.weight.numel():>12,} (frozen)")
print(f"  LoRA trainable:{layer.trainable_parameters():>12,}")
print(f"  Ratio:         {layer.trainable_parameters()/layer.weight.numel():.4%}")

# Verify merge produces same output
merged = layer.merge_weights()
with torch.no_grad():
    out_merged = merged(x)
print(f"\nMerge verification (max diff): {(out - out_merged).abs().max().item():.2e}")

In [ ]:
# Visualize LoRA Parameter Savings Across Ranks
model_dims = [
    ("LLaMA-3.1-8B  (d=4096)", 4096, 32),
    ("LLaMA-3.1-70B (d=8192)", 8192, 80),
    ("GPT-3         (d=12288)", 12288, 96),
]
ranks = [4, 8, 16, 32, 64, 128]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
for name, d, n_layers in model_dims:
    trainable_params = []
    for r in ranks:
        # Assume LoRA on Q, V projections in all layers
        # Q: (d × d), V: (d × d)
        params_per_layer = 2 * r * (d + d)  # 2 matrices, each r*(d+d)
        total = params_per_layer * n_layers
        trainable_params.append(total / 1e6)
    ax.plot(ranks, trainable_params, marker='o', label=name, linewidth=2)

ax.set_xlabel('LoRA Rank (r)')
ax.set_ylabel('Trainable Parameters (millions)')
ax.set_title('LoRA Trainable Parameters vs Rank')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[1]
d = 4096  # LLaMA-3.1-8B
full_params = d * d  # full weight matrix
savings = [100 * (1 - r*(d+d) / full_params) for r in ranks]
ax.bar(range(len(ranks)), savings, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(ranks)))
ax.set_xticklabels([f'r={r}' for r in ranks])
ax.set_ylabel('Parameter Reduction (%)')
ax.set_title('LoRA Savings vs Full Fine-Tuning (d=4096)')
ax.set_ylim(0, 105)
for i, v in enumerate(savings):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../figures/07_lora_parameter_savings.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Applying LoRA with HuggingFace PEFT

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import torch

# For this demo, use a small model. Replace with your target model.
MODEL_NAME = "gpt2"  # small enough to run on CPU
# For real fine-tuning: "meta-llama/Meta-Llama-3-8B", "mistralai/Mistral-7B-v0.1", etc.

# Load base model
print(f"Loading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,  # fp16 or bf16 for larger models
    # For 4-bit quantization (QLoRA), use:
    # quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print(f"Base model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # rank — start with 16, reduce if memory constrained
    lora_alpha=32,           # α — typically 2×rank
    target_modules=["c_attn", "c_proj"],  # which layers to apply LoRA to
    # For LLaMA: ["q_proj", "k_proj", "v_proj", "o_proj"]
    # For Mistral: ["q_proj", "v_proj"] (minimal, fast)
    lora_dropout=0.05,
    bias="none",             # don't train biases
    inference_mode=False,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print parameter summary
model.print_trainable_parameters()

# Inspect which layers have LoRA adapters
print("\nLoRA layers:")
for name, module in model.named_modules():
    if hasattr(module, 'lora_A'):
        A_params = module.lora_A['default'].weight.numel()
        B_params = module.lora_B['default'].weight.numel()
        print(f"  {name}: A={A_params:,} B={B_params:,} total={A_params+B_params:,}")

## 5. Fine-tuning on a Custom Dataset

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset

# Small synthetic dataset — replace with your actual data
train_texts = [
    "The transformer model processes all tokens simultaneously using self-attention.",
    "LoRA reduces the number of trainable parameters by using low-rank decomposition.",
    "Retrieval-augmented generation combines search with language model generation.",
    "KV cache stores key-value pairs from previous tokens to speed up generation.",
    "Quantization reduces model precision from fp32 to int8 or int4 to save memory.",
    "RLHF uses human feedback to align language models with human preferences.",
    "Speculative decoding uses a smaller draft model to propose tokens for verification.",
    "Fine-tuning adapts a pre-trained model to a specific task or domain.",
    "Embeddings are dense vector representations that capture semantic meaning.",
    "Chain-of-thought prompting improves reasoning by asking models to show their work.",
] * 20  # repeat to get more data

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

dataset = Dataset.from_dict({'text': train_texts})
tokenized = dataset.map(tokenize_function, batched=True, remove_columns=['text'])

# Training arguments
training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,  # effective batch = 8
    learning_rate=3e-4,             # LoRA typically uses higher LR than full FT
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=device == 'cuda',          # fp16 on GPU
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Save LoRA adapter only (not the full model)
model.save_pretrained("./lora_adapter")

# Load for inference
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
lora_model = PeftModel.from_pretrained(base_model, "./lora_adapter")

# Option: merge for production deployment (zero inference overhead)
merged_model = lora_model.merge_and_unload()

print(f"LoRA adapter size: {sum(p.numel() for p in lora_model.parameters() if p.requires_grad):,} params")
print(f"Merged model size: {sum(p.numel() for p in merged_model.parameters()):,} params")
print("Merge successful ✅")

## 6. LoRA Hyperparameter Guide

### Rank (r)

| Rank | Use Case | Memory | Quality |
|------|----------|--------|--------|
| 4 | Extremely constrained | Minimal | Basic adaptation |
| 8 | Tight memory budget | Low | Good for simple tasks |
| **16** | **Default / balanced** | **Medium** | **Most use cases** |
| 32 | Complex tasks | Higher | Better for nuanced style |
| 64-128 | Near full-FT quality | High | When quality is critical |

### Target Modules

```python
# Minimal (faster, less memory):
target_modules = ["q_proj", "v_proj"]

# Standard (recommended):
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Full attention + MLP (maximum expressiveness):
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
```

### ⚠️ Engineering Gotchas

1. **Learning rate**: Use 3-10x higher LR for LoRA than full FT (1e-4 to 3e-4)
2. **Alpha/rank ratio**: `alpha = 2*rank` is a safe default; `alpha = rank` is also common
3. **Dropout**: Set to 0.0 for small datasets to avoid underfitting
4. **Save only adapters**: Don't save the full model — adapters are tiny (a few MB!)
5. **Gradient checkpointing**: Enable with `model.enable_input_require_grads()` to reduce memory

## 7. Exercises

1. **Rank Sweep**: Train the same model with r=4, 8, 16, 32, 64. Plot validation loss vs rank and parameter count.

2. **Target Module Ablation**: Compare `[q_proj, v_proj]` vs `[q, k, v, o]` vs adding MLP layers. Which gives the best quality/parameter trade-off?

3. **QLoRA**: Modify the code to use 4-bit quantization (`BitsAndBytesConfig`) before applying LoRA. Measure memory reduction vs quality change.

4. **Multi-Task Adapters**: Train two separate LoRA adapters for two different tasks. Show that you can swap them on the same base model.

5. **Custom LoRA Layer**: Extend `LoRALinear` to support `lora_dropout` and verify the output matches PEFT's LoRA on a small model.

---

## 8. References

- [Hu et al. (2022) — LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- [Dettmers et al. (2023) — QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314)
- [HuggingFace PEFT Documentation](https://huggingface.co/docs/peft/)
- [Sebastian Raschka — LoRA from scratch](https://lightning.ai/pages/community/lora-insights/)